In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('2019-Oct_funnel-smartphone.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart
0,2019-10-01 09:00:04,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
1,2019-10-01 09:00:11,view,1004545,2053013555631882655,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb,False,9,1,Tuesday,오전 (6-12시),False,0,0
2,2019-10-01 09:00:11,view,1005011,2053013555631882655,electronics.smartphone,samsung,900.64,530282093,50a293fb-5940-41b2-baf3-17af0e812101,False,9,1,Tuesday,오전 (6-12시),False,0,0
3,2019-10-01 09:00:19,view,1005135,2053013555631882655,electronics.smartphone,apple,1747.79,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
4,2019-10-01 09:00:20,view,1003306,2053013555631882655,electronics.smartphone,apple,588.77,555446831,6ec635da-ea15-4a5d-96b4-c8ca9d38f89f,False,9,1,Tuesday,오전 (6-12시),False,0,0


In [3]:
df = df.sort_values(['user_session', 'event_time']).copy()

In [4]:
grouped = df.groupby(['user_session', 'product_id'])

In [5]:
event_first_time = (
    df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

funnel_df['segment'] = np.select(
    [
        funnel_df['view_to_cart_to_purchase'],  # 완전 전환
        
        funnel_df['view_to_cart'] & ~funnel_df['view_to_cart_to_purchase'],  # cart까지 갔다가 이탈
        
        funnel_df['view'] & ~funnel_df['view_to_cart']  # view만 하고 이탈
    ],
    [
        'conversion',
        'view_cart_drop',
        'view_only_drop'
    ],
    default='etc'
)

In [6]:
segment_summary = (
    funnel_df['segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = ['segment', 'count']

segment_summary['ratio'] = (
    segment_summary['count'] / segment_summary['count'].sum() * 100
)

segment_summary

,segment,count,ratio
0,view_only_drop,6694294,94.820109
1,conversion,191430,2.711475
2,view_cart_drop,172935,2.449506
3,etc,1335,0.018909


In [7]:
funnel_df['segment'].value_counts(normalize=True) * 100

segment
view_only_drop    94.820109
conversion         2.711475
view_cart_drop     2.449506
etc                0.018909
Name: proportion, dtype: float64

In [8]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary

,step,count
0,view,7058659
1,view_to_cart,364365
2,view_to_cart_to_purchase,191430


In [9]:
# 1. 구매 전환 케이스
conversion_df = funnel_df[
    funnel_df['segment'] == 'conversion'
].copy()

# 2. view만 하고 이탈한 케이스
view_only_drop_df = funnel_df[
    funnel_df['segment'] == 'view_only_drop'
].copy()

# 3. view → cart 후 구매 없이 이탈한 케이스
view_cart_drop_df = funnel_df[
    funnel_df['segment'] == 'view_cart_drop'
].copy()

In [10]:
df_segment = df.merge(
    funnel_df[['user_session', 'product_id', 'segment']],
    on=['user_session', 'product_id'],
    how='left'
)

df_segment.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart,segment
0,2019-10-31 15:23:12,view,1005115,2053013555631882655,electronics.smartphone,apple,955.84,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
1,2019-10-31 15:23:52,view,1005105,2053013555631882655,electronics.smartphone,apple,1349.46,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
2,2019-10-31 15:25:30,view,1005105,2053013555631882655,electronics.smartphone,apple,1349.46,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
3,2019-10-31 15:26:58,view,1004858,2053013555631882655,electronics.smartphone,samsung,131.53,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
4,2019-10-31 15:28:21,view,1005104,2053013555631882655,electronics.smartphone,apple,993.27,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop


이탈 집단은 view 시점의 price / price 이상치 처리..

In [11]:
segment_price_detail = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby('segment')
    .agg(
        view_count=('price', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        min_price=('price', 'min'),
        max_price=('price', 'max')
    )
    .reset_index()
)

segment_price_detail

,segment,view_count,avg_price,median_price,min_price,max_price
0,conversion,473525,430.082983,250.82,0.0,2110.45
1,view_cart_drop,454334,426.618817,250.69,0.0,2110.45
2,view_only_drop,9690789,478.870885,287.97,0.0,2110.45


In [12]:
conversion_purchase_price = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby('segment')
    .agg(
        purchase_count=('price', 'count'),
        avg_purchase_price=('price', 'mean'),
        median_purchase_price=('price', 'median'),
        min_purchase_price=('price', 'min'),
        max_purchase_price=('price', 'max')
    )
    .reset_index()
)

conversion_purchase_price

,segment,purchase_count,avg_purchase_price,median_purchase_price,min_purchase_price,max_purchase_price
0,conversion,209279,434.001041,250.82,38.3,2110.45


세그먼트 별 브랜드 분포

In [13]:
segment_brand = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby(['segment', 'brand'])
    .size()
    .reset_index(name='count')
    .sort_values(['segment', 'count'], ascending=[True, False])
)

segment_brand.head(20)

,segment,brand,count
19,conversion,samsung,215893
0,conversion,apple,146055
25,conversion,xiaomi,49824
9,conversion,huawei,35770
17,conversion,oppo,19006
24,conversion,vivo,2857
13,conversion,meizu,763
7,conversion,honor,636
14,conversion,nokia,611
20,conversion,sony,409


In [14]:
segment_brand['segment_total'] = (
    segment_brand.groupby('segment')['count'].transform('sum')
)

segment_brand['ratio'] = (
    segment_brand['count'] / segment_brand['segment_total'] * 100
)

segment_brand.sort_values(['segment', 'ratio'], ascending=[True, False]).head(30)

,segment,brand,count,segment_total,ratio
19,conversion,samsung,215893,473525,45.592735
0,conversion,apple,146055,473525,30.844200
25,conversion,xiaomi,49824,473525,10.521937
9,conversion,huawei,35770,473525,7.553983
17,conversion,oppo,19006,473525,4.013727
24,conversion,vivo,2857,473525,0.603347
13,conversion,meizu,763,473525,0.161132
7,conversion,honor,636,473525,0.134312
14,conversion,nokia,611,473525,0.129032
20,conversion,sony,409,473525,0.086373


상위 구매 / 이탈집단 상위 조회

In [15]:
segment_product_purchase = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
)

segment_product_purchase.head(10)

,segment,product_id,revenue,purchase_count
412,conversion,1005115,6138721.37,6219
402,conversion,1005105,6016312.64,4283
129,conversion,1004249,4145501.93,5596
246,conversion,1004767,3700130.24,14855
432,conversion,1005135,3320724.71,1919
8,conversion,1002544,3200144.75,6952
289,conversion,1004856,2883622.98,21965
1,conversion,1002524,2216241.15,4148
298,conversion,1004870,2114322.50,7412
22,conversion,1003306,1689608.85,2896


In [16]:
segment_product_view = (
    df_segment[
        (df_segment['segment'].isin(['view_only_drop', 'view_cart_drop'])) &
        (df_segment['event_type'] == 'view')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        view_count=('product_id', 'size'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median')
    )
    .reset_index()
    .sort_values(['segment', 'view_count'], ascending=[True, False])
)

segment_product_view.head(20)

,segment,product_id,view_count,avg_price,median_price
305,view_cart_drop,1004856,33276,131.296034,131.53
262,view_cart_drop,1004767,28268,248.988366,250.13
314,view_cart_drop,1004870,15211,285.292802,285.40
290,view_cart_drop,1004833,14439,171.993932,172.15
134,view_cart_drop,1004249,12497,740.798575,741.06
429,view_cart_drop,1005115,11991,987.839528,992.05
249,view_cart_drop,1004741,11876,190.090504,190.22
7,view_cart_drop,1002544,11198,460.302427,460.11
293,view_cart_drop,1004836,10795,228.605805,229.42
247,view_cart_drop,1004739,9884,191.838414,190.21


세그먼트 별 조회 상품수

In [17]:
views_segment = df_segment[df_segment['event_type'] == 'view'].copy()

session_view_depth_segment = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        view_event_count=('product_id', 'size'),
        unique_view_products=('product_id', 'nunique')
    )
    .reset_index()
)

segment_view_depth = (
    session_view_depth_segment.groupby('segment')
    .agg(
        avg_view_events_per_session=('view_event_count', 'mean'),
        median_view_events_per_session=('view_event_count', 'median'),
        avg_unique_products_per_session=('unique_view_products', 'mean'),
        median_unique_products_per_session=('unique_view_products', 'median')
    )
    .reset_index()
)

segment_view_depth

,segment,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session
0,conversion,2.627454,2.0,1.062190,1.0
1,view_cart_drop,2.800607,2.0,1.066006,1.0
2,view_only_drop,3.333653,2.0,2.302852,1.0


세그먼트별 동일 세션 내 모델 비교 깊이

In [18]:
segment_compare_depth = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        unique_models_compared=('product_id', 'nunique')
    )
    .reset_index()
)

segment_compare_summary = (
    segment_compare_depth.groupby('segment')
    .agg(
        avg_model_compare_depth=('unique_models_compared', 'mean'),
        median_model_compare_depth=('unique_models_compared', 'median'),
        max_model_compare_depth=('unique_models_compared', 'max')
    )
    .reset_index()
)

segment_compare_ratio = (
    segment_compare_depth.groupby('segment')
    .apply(lambda g: pd.Series({
        'compare_2plus_session_ratio': (g['unique_models_compared'] >= 2).mean() * 100,
        'compare_3plus_session_ratio': (g['unique_models_compared'] >= 3).mean() * 100
    }))
    .reset_index()
)

segment_compare_summary = segment_compare_summary.merge(
    segment_compare_ratio,
    on='segment',
    how='left'
)

segment_compare_summary

,segment,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,1.062190,1.0,10,5.276825,0.721333
1,view_cart_drop,1.066006,1.0,12,5.566274,0.765594
2,view_only_drop,2.302852,1.0,120,41.788426,24.833933


In [19]:
final_segment_compare = (
    segment_price_detail
    .merge(segment_view_depth, on='segment', how='left')
    .merge(segment_compare_summary, on='segment', how='left')
)

final_segment_compare

,segment,view_count,avg_price,median_price,min_price,max_price,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,473525,430.082983,250.82,0.0,2110.45,2.627454,2.0,1.062190,1.0,1.062190,1.0,10,5.276825,0.721333
1,view_cart_drop,454334,426.618817,250.69,0.0,2110.45,2.800607,2.0,1.066006,1.0,1.066006,1.0,12,5.566274,0.765594
2,view_only_drop,9690789,478.870885,287.97,0.0,2110.45,3.333653,2.0,2.302852,1.0,2.302852,1.0,120,41.788426,24.833933
